# Train the NAFNet denoiser (free GPU)

Runs the training in this repo on a Colab T4, which takes about an hour
for the `nano` architecture. Training on a laptop CPU is not practical.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

At the end you download `nafnet.onnx` and copy it to the Jetson's
`models/` folder.

---

## About the data

The denoiser learns from **pairs**: a clean frame and a degraded copy of
it. The degradation is simulated (blur, sensor noise, low light, JPEG) and
that part is faithful to real cameras. The *clean* side has to come from
somewhere real if you want the model to work on real footage.

Cell 3 offers two sources. Use the synthetic one to get a model today;
replace it with your own footage before you trust the numbers.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'

In [ ]:
#@title 2. Get the code
!git clone -q https://github.com/Indumini2005/Re-id-techthrive.git
%cd Re-id-techthrive
!pip install -q onnx onnxsim onnxruntime
!ls

In [ ]:
#@title 3a. Data option A - synthetic CCTV scenes (no footage needed)
# Procedural rooms with people at depth-correct scale. Good enough to
# validate the whole chain and to warm-start; weaker than real frames.
!python -m training.synthesize_cctv frames \
    --out data/synth --count 2000 --width 960 --height 540

from IPython.display import Image, display
display(Image('data/synth/clean/000000.png', width=640))

In [ ]:
#@title 3b. Data option B - your own footage (strongly preferred)
# Record 10-15 minutes from a phone propped high in a corridor or doorway,
# angled down, people walking through. That is the real thing and beats any
# simulation. Upload the clips, then:
#
#   from google.colab import files; files.upload()
#   !python -m training.make_dataset video --src . --out data/real --fps 2
#
# Frames below the sharpness threshold are dropped automatically, so blurry
# handheld moments do not poison the clean targets.
print('skip this cell if you are using the synthetic data above')

In [ ]:
#@title 4. Preview the degradation the model will learn to undo
!python -m training.degradations data/synth/clean/000000.png \
    --preset cctv --n 4 --out preview.png

from IPython.display import Image, display
display(Image('preview.png', width=1100))
# clean | four degraded variants. If these look nothing like your cameras,
# tune training/degradations.py before spending an hour on training.

In [ ]:
#@title 5. Train
#@markdown `nano` is what the Jetson Nano can actually run. `small` is ~4x
#@markdown the cost for roughly +0.5 dB; measure before choosing it.
ARCH = 'nano'      #@param ['nano', 'small', 'base']
EPOCHS = 80        #@param {type:'integer'}
PATCH = 128        #@param {type:'integer'}
BATCH = 16         #@param {type:'integer'}
DATA = 'data/synth/clean'  #@param {type:'string'}

!python -m training.train_denoiser \
    --train_dir {DATA} --arch {ARCH} --preset cctv \
    --epochs {EPOCHS} --patch_size {PATCH} --batch_size {BATCH} \
    --repeat 4 --workers 2 --name nafnet

In [ ]:
#@title 6. Look at what it learned
import glob
from IPython.display import Image, display

samples = sorted(glob.glob('runs/nafnet/samples/*.png'))
print('noisy | denoised | clean')
for path in samples[-2:]:
    print(path)
    display(Image(path, width=900))

import pandas as pd
log = pd.read_csv('runs/nafnet/log.csv')
display(log.tail(10))
log.plot(x='epoch', y='val_psnr', figsize=(7, 3), title='validation PSNR (dB)');

In [ ]:
#@title 7. Score it
!python -m training.evaluate --ckpt runs/nafnet/best.pth \
    --data_dir data/synth/clean --limit 40 --max_side 720
# Under about +3 dB means the degradation preset does not match the data.
# Retune degradations.py rather than training for longer.

In [ ]:
#@title 8. Export ONNX and download
#@markdown opset 12 keeps JetPack 4.6's ONNX Runtime 1.11 happy.
!python export/export_onnx.py --ckpt runs/nafnet/best.pth \
    --tile 128 --opset 12 --simplify

# Confirm the exported graph matches the checkpoint before trusting it
!python -m training.evaluate --onnx models/nafnet.onnx \
    --data_dir data/synth/clean --limit 20 --max_side 720

from google.colab import files
files.download('models/nafnet.onnx')

## Then on the Jetson

```bash
# copy nafnet.onnx into models/, then
python3 check_device.py
python3 denoise.py --input_dir some_frames --output_dir out --tile 128 --compare
```

`--compare` writes noisy-vs-denoised strips so you can judge it by eye on
real frames, which is the only test that counts.

Optionally build a TensorRT engine **on the device** (FP16 only — Maxwell
has no INT8):

```bash
python3 export/export_tensorrt.py --onnx models/nafnet.onnx --precision fp16 --benchmark
```